<a href="https://colab.research.google.com/github/AdilM01/AI-Projects/blob/feature%2Fai-projects/Progressive%20Fine-Tuning%20Framework%20for%20Malware%20Image%20Classification%20%20%20A%20Comparative%20Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"adiiml",\r\n"key":"6ef5d41583443108686811b0d219bd33"}'}

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d manmandes/malimg
!unzip malimg.zip

Streaming output truncated to the last 5000 lines.
  inflating: malimg_dataset/train/Allaple.L/057ad485bd587c574fc45b30e9c178ff.png  
  inflating: malimg_dataset/train/Allaple.L/057c63ce597787d48704bae075a2dfee.png  
  inflating: malimg_dataset/train/Allaple.L/057e01696b4916e12faeccf3ba98f34e.png  
  inflating: malimg_dataset/train/Allaple.L/058225c6a8c62081e70d077adb7d948e.png  
  inflating: malimg_dataset/train/Allaple.L/05826101eb5936bcae5fbbeaddcd2a36.png  
  inflating: malimg_dataset/train/Allaple.L/0583529f509164befa6236014dd6c504.png  
  inflating: malimg_dataset/train/Allaple.L/05851d6bd563c266d26386488f72d45b.png  
  inflating: malimg_dataset/train/Allaple.L/058910e8a2c1e23dd9b140572aed632d.png  
  inflating: malimg_dataset/train/Allaple.L/058aee7b24181a28532ac332a133d6be.png  
  inflating: malimg_dataset/train/Allaple.L/058c343d9f8c0a3524414cf00c7cb318.png  
  inflating: malimg_dataset/train/Allaple.L/058cd19616bb53ececc3dc67336227ab.png  
  inflating: malimg_dataset/train/Al

In [ ]:
"""
================================================================================
Progressive Fine-Tuning Framework for Malware Image Classification
A Comparative Study
================================================================================
Models (6 total — VGG16 removed):
  ┌─────────────────────┬──────────────┬─────────────────────────────────────┐
  │ Model               │ Source       │ IEEE Relevance                      │
  ├─────────────────────┼──────────────┼─────────────────────────────────────┤
  │ ResNet50            │ torchvision  │ Baseline CNN, ubiquitous in IEEE    │
  │ ResNet101           │ torchvision  │ Deeper residual, strong benchmark   │
  │ DenseNet121         │ torchvision  │ Feature reuse, high cite count      │
  │ MobileNetV2         │ torchvision  │ Edge/IoT deployment comparison      │
  │ InceptionResNetV2   │ timm         │ Deep hybrid, malware sota           │
  │ Xception            │ timm         │ Depthwise separable, IEEE papers    │
  └─────────────────────┴──────────────┴─────────────────────────────────────┘

Layer Division (principled, per-module not per-parameter):
  BASE  → bottom 33% of named modules  (LR = 1e-5)
  TORSO → middle 33%                   (LR = 5e-5)
  UPPER → top 34%                      (LR = 1e-4)
  HEAD  → custom classifier            (LR = 1e-3)

Metrics Suite v4:
  ⏱  Time    : inference time/image, feature extraction time, model load time
  💾 Resource : GPU usage (%), VRAM (MB), RAM footprint (MB)
  📊 Performance : Accuracy, Precision, Recall, F1, AUC, Confusion Matrix,
                   MCC, Cohen's Kappa, Specificity, Balanced Accuracy,
                   Top-5 Accuracy, Brier Score

================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
#  ★  GOOGLE DRIVE PATH  —
# ─────────────────────────────────────────────────────────────────────────────
DRIVE_CONFIG = {

    "DRIVE_FOLDER"     : "MalwareExperiments",


    "DRIVE_SUBFOLDER"  : "",
    "SAVE_TO_DRIVE"    : True,


    "INCREMENTAL_SAVE" : True,
}
# ─────────────────────────────────────────────────────────────────────────────

import os, time, copy, json, warnings, psutil
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.models import (
    resnet50,    ResNet50_Weights,
    resnet101,   ResNet101_Weights,
    densenet121, DenseNet121_Weights,
    mobilenet_v2, MobileNet_V2_Weights,
    inception_v3, Inception_V3_Weights,
)

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, classification_report,
    matthews_corrcoef, cohen_kappa_score,
    balanced_accuracy_score, roc_curve,
    top_k_accuracy_score, brier_score_loss,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from sklearn.calibration import calibration_curve
from scipy.stats import friedmanchisquare

try:
    from scikit_posthocs import posthoc_nemenyi_friedman
    POSTHOCS_AVAILABLE = True
except ImportError:
    POSTHOCS_AVAILABLE = False

try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except ImportError:
    THOP_AVAILABLE = False

try:
    import pynvml
    pynvml.nvmlInit()
    PYNVML_AVAILABLE = True
except Exception:
    PYNVML_AVAILABLE = False

warnings.filterwarnings('ignore')


# ─────────────────────────────────────────────────────────────────────────────
# GOOGLE DRIVE SAVE UTILITY
# ─────────────────────────────────────────────────────────────────────────────
def mount_and_save_to_drive(local_save_dir: Path) -> str | None:
    """
    Mounts Google Drive in Colab and copies all experiment outputs.

    Drive path is fully controlled by DRIVE_CONFIG at the top of this file:
        MyDrive / <DRIVE_FOLDER> / <DRIVE_SUBFOLDER or timestamp> / *

    Returns the Drive destination path string, or None if not in Colab.
    """
    if not DRIVE_CONFIG["SAVE_TO_DRIVE"]:
        return None
    try:
        from google.colab import drive as colab_drive
        import shutil

        drive_mount = Path("/content/drive")
        if not drive_mount.exists() or not (drive_mount / "MyDrive").exists():
            print("\n  Mounting Google Drive …")
            colab_drive.mount("/content/drive", force_remount=False)

        subfolder = DRIVE_CONFIG["DRIVE_SUBFOLDER"].strip() or local_save_dir.name
        dest = drive_mount / "MyDrive" / DRIVE_CONFIG["DRIVE_FOLDER"] / subfolder
        dest.mkdir(parents=True, exist_ok=True)

        copied = 0
        for f in local_save_dir.rglob("*"):
            if f.is_file():
                target = dest / f.relative_to(local_save_dir)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)
                copied += 1

        print(f"  ✓ Saved {copied} file(s) → "
              f"MyDrive/{DRIVE_CONFIG['DRIVE_FOLDER']}/{subfolder}")
        return str(dest)

    except ImportError:
        print("  [Drive] Not running in Google Colab — skipping Drive upload.")
        return None
    except Exception as e:
        print(f"  [Drive] Upload failed: {e}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    "data_dir"            : "./malimg_dataset",
    "num_classes"         : 25,
    "img_size"            : 224,
    "batch_size"          : 32,
    "num_workers"         : 4,

    # Epochs per stage
    "stage1_epochs"       : 15,
    "stage2_epochs"       : 20,
    "stage3_epochs"       : 25,

    # Differential LRs
    "lr_head"             : 1e-3,
    "lr_upper"            : 1e-4,
    "lr_torso"            : 5e-5,
    "lr_base"             : 1e-5,

    # Regularization
    "weight_decay"        : 1e-4,
    "dropout"             : 0.4,
    "label_smoothing"     : 0.1,

    # Early stopping
    "early_stop_patience" : 7,
    "early_stop_delta"    : 5e-4,

    # Warmup
    "stage_warmup_epochs" : 2,

    # Gradient clipping
    "grad_clip"           : 1.0,

    "device"              : "cuda" if torch.cuda.is_available() else "cpu",
    "use_amp"             : True,
    "seed"                : 42,

    "save_dir"            : "./experiment_results",
    "checkpoint_dir"      : "./checkpoints",
}

MODELS_TO_RUN = [
    "resnet50",
    "resnet101",
    "densenet121",
    "mobilenet_v2",
    "inception_resnetv2",
    "xception",
]


# ─────────────────────────────────────────────────────────────────────────────
# REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])


# ─────────────────────────────────────────────────────────────────────────────
# DATA PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def get_transforms(img_size, phase):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if phase == "train":
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.2),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def build_dataloaders(cfg):
    train_ds = datasets.ImageFolder(
        os.path.join(cfg["data_dir"], "train"),
        transform=get_transforms(cfg["img_size"], "train"))
    val_ds = datasets.ImageFolder(
        os.path.join(cfg["data_dir"], "val"),
        transform=get_transforms(cfg["img_size"], "val"))
    test_ds = datasets.ImageFolder(
        os.path.join(cfg["data_dir"], "test"),
        transform=get_transforms(cfg["img_size"], "val"))

    counts  = np.bincount(train_ds.targets)
    weights = 1.0 / torch.tensor(counts, dtype=torch.float)
    sampler = WeightedRandomSampler(weights[train_ds.targets], len(train_ds))

    kw = dict(num_workers=cfg["num_workers"], pin_memory=True, drop_last=True)
    train_loader = DataLoader(train_ds, cfg["batch_size"], sampler=sampler, **kw)
    val_loader   = DataLoader(val_ds,   cfg["batch_size"], shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  cfg["batch_size"], shuffle=False, **kw)
    return train_loader, val_loader, test_loader, train_ds.classes


# ─────────────────────────────────────────────────────────────────────────────
# CLASSIFIER HEAD
# ─────────────────────────────────────────────────────────────────────────────
def build_classifier_head(feature_dim: int, num_classes: int, dropout: float):
    return nn.Sequential(
        nn.BatchNorm1d(feature_dim),
        nn.Dropout(dropout),
        nn.Linear(feature_dim, 512),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(512),
        nn.Dropout(dropout * 0.5),
        nn.Linear(512, num_classes),
    )


# ─────────────────────────────────────────────────────────────────────────────
# MODEL FACTORY  (VGG16 removed)
# ─────────────────────────────────────────────────────────────────────────────
class ModelBundle:
    def __init__(self, model, classifier, base_modules, torso_modules, upper_modules):
        self.model         = model
        self.classifier    = classifier
        self.base_modules  = base_modules
        self.torso_modules = torso_modules
        self.upper_modules = upper_modules


def _split_module_names(backbone: nn.Module):
    children = list(backbone.named_children())
    n  = len(children)
    b_end = n // 3;  t_end = 2 * n // 3
    base  = children[:b_end]
    torso = children[b_end:t_end]
    upper = children[t_end:]
    print(f"  Module split → BASE: {len(base)} | TORSO: {len(torso)} | UPPER: {len(upper)}")
    for g, nm in [("BASE", base), ("TORSO", torso), ("UPPER", upper)]:
        print(f"    {g}: {[n for n,_ in nm]}")
    return base, torso, upper


def _freeze_all(module: nn.Module):
    for p in module.parameters(): p.requires_grad = False

def _unfreeze_module_list(module_list):
    for _, m in module_list:
        for p in m.parameters(): p.requires_grad = True

def _params_of(module_list):
    params = []
    for _, m in module_list:
        params += [p for p in m.parameters() if p.requires_grad]
    return params


class ModelFactory:
    @staticmethod
    def build(name: str, num_classes: int, dropout: float, device: str):
        name = name.lower()
        if   name == "resnet50":         return ModelFactory._build_resnet(resnet50(weights=ResNet50_Weights.IMAGENET1K_V2),    num_classes, dropout, device)
        elif name == "resnet101":        return ModelFactory._build_resnet(resnet101(weights=ResNet101_Weights.IMAGENET1K_V2),  num_classes, dropout, device)
        elif name == "densenet121":      return ModelFactory._build_densenet(densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1), num_classes, dropout, device)
        elif name == "mobilenet_v2":     return ModelFactory._build_mobilenet(mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2), num_classes, dropout, device)
        elif name in ("inception_resnetv2", "xception"): return ModelFactory._build_timm(name, num_classes, dropout, device)
        else: raise ValueError(f"Unknown model: {name}")

    @staticmethod
    def _build_resnet(bb, num_classes, dropout, device):
        feat_dim = bb.fc.in_features
        bb.fc = nn.Identity()
        _freeze_all(bb)
        class ResNetWrapper(nn.Module):
            def __init__(self, backbone, head):
                super().__init__()
                self.backbone   = backbone
                self.flatten    = nn.Flatten()
                self.classifier = head
            def forward(self, x):
                return self.classifier(self.flatten(self.backbone(x)))
        head  = build_classifier_head(feat_dim, num_classes, dropout)
        model = ResNetWrapper(bb, head).to(device)
        base, torso, upper = _split_module_names(bb)
        return ModelBundle(model, model.classifier, base, torso, upper)

    @staticmethod
    def _build_densenet(bb, num_classes, dropout, device):
        feat_dim = bb.classifier.in_features
        bb.classifier = nn.Identity()
        _freeze_all(bb)
        class DenseNetWrapper(nn.Module):
            def __init__(self, backbone, head):
                super().__init__()
                self.backbone   = backbone
                self.pool       = nn.AdaptiveAvgPool2d(1)
                self.flatten    = nn.Flatten()
                self.classifier = head
            def forward(self, x):
                feat = self.backbone.features(x)
                feat = nn.functional.relu(feat, inplace=True)
                feat = self.pool(feat); feat = self.flatten(feat)
                return self.classifier(feat)
        head  = build_classifier_head(feat_dim, num_classes, dropout)
        model = DenseNetWrapper(bb, head).to(device)
        base, torso, upper = _split_module_names(bb.features)
        return ModelBundle(model, model.classifier, base, torso, upper)

    @staticmethod
    def _build_mobilenet(bb, num_classes, dropout, device):
        feat_dim = bb.last_channel
        bb.classifier = nn.Identity()
        _freeze_all(bb)
        class MobileNetWrapper(nn.Module):
            def __init__(self, backbone, head):
                super().__init__()
                self.features   = backbone.features
                self.pool       = nn.AdaptiveAvgPool2d(1)
                self.flatten    = nn.Flatten()
                self.classifier = head
            def forward(self, x):
                return self.classifier(self.flatten(self.pool(self.features(x))))
        head  = build_classifier_head(feat_dim, num_classes, dropout)
        model = MobileNetWrapper(bb, head).to(device)
        base, torso, upper = _split_module_names(bb.features)
        return ModelBundle(model, model.classifier, base, torso, upper)

    @staticmethod
    def _build_timm(name, num_classes, dropout, device):
        try:
            import timm
            arch = "inception_resnet_v2" if name == "inception_resnetv2" else "xception"
            bb = timm.create_model(arch, pretrained=True, num_classes=0, global_pool='avg')
            feat_dim = bb.num_features
            _freeze_all(bb)
            class TimmWrapper(nn.Module):
                def __init__(self, backbone, head):
                    super().__init__()
                    self.backbone   = backbone
                    self.classifier = head
                def forward(self, x):
                    return self.classifier(self.backbone(x))
            head  = build_classifier_head(feat_dim, num_classes, dropout)
            model = TimmWrapper(bb, head).to(device)
            base, torso, upper = _split_module_names(bb)
            return ModelBundle(model, model.classifier, base, torso, upper)
        except ImportError:
            print(f"timm not found. Falling back to InceptionV3 for {name}.")
            bb = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=False)
            feat_dim = bb.fc.in_features; bb.fc = nn.Identity(); _freeze_all(bb)
            class InceptionWrapper(nn.Module):
                def __init__(self, backbone, head):
                    super().__init__()
                    self.backbone   = backbone
                    self.flatten    = nn.Flatten()
                    self.classifier = head
                def forward(self, x):
                    out = self.backbone(x)
                    if isinstance(out, tuple): out = out[0]
                    return self.classifier(self.flatten(out) if out.dim() > 2 else out)
            head  = build_classifier_head(feat_dim, num_classes, dropout)
            model = InceptionWrapper(bb, head).to(device)
            base, torso, upper = _split_module_names(bb)
            return ModelBundle(model, model.classifier, base, torso, upper)


# ─────────────────────────────────────────────────────────────────────────────
# PROGRESSIVE UNFREEZER
# ─────────────────────────────────────────────────────────────────────────────
class ProgressiveUnfreezer:
    def __init__(self, bundle: ModelBundle, cfg: dict):
        self.bundle = bundle
        self.cfg    = cfg

    def build_optimizer(self, stage: int):
        cfg = self.cfg
        if stage == 1:
            param_groups = [{"params": list(self.bundle.classifier.parameters()),
                             "lr": cfg["lr_head"], "name": "classifier"}]
        elif stage == 2:
            _unfreeze_module_list(self.bundle.upper_modules)
            _unfreeze_module_list(self.bundle.torso_modules)
            param_groups = [
                {"params": _params_of(self.bundle.upper_modules),   "lr": cfg["lr_upper"], "name": "upper"},
                {"params": _params_of(self.bundle.torso_modules),   "lr": cfg["lr_torso"], "name": "torso"},
                {"params": list(self.bundle.classifier.parameters()),"lr": cfg["lr_head"],  "name": "classifier"},
            ]
        elif stage == 3:
            _unfreeze_module_list(self.bundle.base_modules)
            param_groups = [
                {"params": _params_of(self.bundle.base_modules),    "lr": cfg["lr_base"],  "name": "base"},
                {"params": _params_of(self.bundle.torso_modules),   "lr": cfg["lr_torso"], "name": "torso"},
                {"params": _params_of(self.bundle.upper_modules),   "lr": cfg["lr_upper"], "name": "upper"},
                {"params": list(self.bundle.classifier.parameters()),"lr": cfg["lr_head"],  "name": "classifier"},
            ]
        param_groups = [g for g in param_groups if len(g["params"]) > 0]
        optimizer = optim.AdamW(param_groups, weight_decay=cfg["weight_decay"])
        trainable = sum(p.numel() for g in param_groups for p in g["params"])
        print(f"  Stage {stage} → Groups: {[g['name'] for g in param_groups]} | Trainable: {trainable:,}")
        return optimizer

    def trainable_count(self):
        return sum(p.numel() for p in self.bundle.model.parameters() if p.requires_grad)


# ─────────────────────────────────────────────────────────────────────────────
# EARLY STOPPING
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=7, min_delta=5e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self._reset()

    def _reset(self):
        self.counter = 0; self.best_score = None
        self.best_weights = None; self.stopped = False

    def reset_for_new_stage(self, model):
        self._reset()
        self.best_weights = copy.deepcopy(model.state_dict())

    def step(self, score, model):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score; self.counter = 0
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stopped = True
        return self.stopped

    def restore(self, model):
        if self.best_weights: model.load_state_dict(self.best_weights)


# ─────────────────────────────────────────────────────────────────────────────
# RESOURCE MONITOR  💾
# ─────────────────────────────────────────────────────────────────────────────
class ResourceMonitor:
    """
    Snapshots GPU %, VRAM (MB), and RAM (MB) at call time.
    Works with or without pynvml.
    """
    @staticmethod
    def snapshot(device: str) -> dict:
        ram_mb = psutil.Process(os.getpid()).memory_info().rss / 1024**2

        vram_mb   = 0.0
        gpu_util  = 0.0

        if device == "cuda":
            # VRAM via PyTorch (always available)
            vram_mb = torch.cuda.memory_allocated() / 1024**2

            # GPU utilisation via pynvml (optional)
            if PYNVML_AVAILABLE:
                try:
                    handle = pynvml.nvmlDeviceGetHandleByIndex(
                        torch.cuda.current_device())
                    util = pynvml.nvmlDeviceGetUtilizationRates(handle)
                    gpu_util = float(util.gpu)
                    vram_mb  = pynvml.nvmlDeviceGetMemoryInfo(handle).used / 1024**2
                except Exception:
                    pass

        return {"gpu_util_pct": gpu_util, "vram_mb": vram_mb, "ram_mb": ram_mb}


# ─────────────────────────────────────────────────────────────────────────────
# METRICS TRACKER
# ─────────────────────────────────────────────────────────────────────────────
class MetricsTracker:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.history     = defaultdict(list)

    def compute(self, loss, preds, labels, probs):
        arr_p  = np.array(preds)
        arr_l  = np.array(labels)
        arr_pr = np.array(probs)

        acc      = accuracy_score(arr_l, arr_p)
        f1       = f1_score(arr_l, arr_p, average='macro', zero_division=0)
        prec     = precision_score(arr_l, arr_p, average='macro', zero_division=0)
        rec      = recall_score(arr_l, arr_p, average='macro', zero_division=0)
        bal_acc  = balanced_accuracy_score(arr_l, arr_p)
        mcc      = matthews_corrcoef(arr_l, arr_p)
        kappa    = cohen_kappa_score(arr_l, arr_p)

        cm = confusion_matrix(arr_l, arr_p)
        spec_list = []
        for i in range(len(cm)):
            tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
            fp = cm[:,i].sum() - cm[i,i]
            spec_list.append(tn / (tn + fp + 1e-9))
        specificity = float(np.mean(spec_list))

        try:
            lb  = label_binarize(arr_l, classes=list(range(self.num_classes)))
            auc = roc_auc_score(lb, arr_pr, multi_class='ovr', average='macro')
        except Exception:
            auc = 0.0
        try:
            top5 = top_k_accuracy_score(arr_l, arr_pr, k=min(5, self.num_classes))
        except Exception:
            top5 = acc
        try:
            lb    = label_binarize(arr_l, classes=list(range(self.num_classes)))
            brier = float(np.mean([brier_score_loss(lb[:,c], arr_pr[:,c])
                                   for c in range(self.num_classes)]))
        except Exception:
            brier = 1.0

        return {
            "loss": loss, "acc": acc, "f1": f1, "prec": prec, "rec": rec,
            "auc": auc, "bal_acc": bal_acc, "mcc": mcc, "kappa": kappa,
            "specificity": specificity, "top5_acc": top5, "brier": brier,
        }

    def log(self, phase, stage, epoch, metrics, resource: dict, lat_ms: float):
        row = {"stage": stage, "epoch": epoch, "lat_ms": lat_ms, **resource, **metrics}
        for k, v in row.items():
            self.history[f"{phase}_{k}"].append(v)
        return row


# ─────────────────────────────────────────────────────────────────────────────
# EPOCH RUNNER
# ─────────────────────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, criterion, scaler, device, phase):
    model.train(phase == "train")
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    t0 = time.perf_counter()

    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with autocast(device_type=device, enabled=(device == "cuda")):
            logits = model(X)
            loss   = criterion(logits, y)
        if phase == "train":
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        probs = torch.softmax(logits.detach(), dim=1)
        preds = logits.detach().argmax(1)
        total_loss  += loss.item() * X.size(0)
        all_preds   += preds.cpu().tolist()
        all_labels  += y.cpu().tolist()
        all_probs   += probs.cpu().tolist()

    lat_ms   = (time.perf_counter() - t0) * 1000 / len(loader)
    resource = ResourceMonitor.snapshot(device)
    avg_loss = total_loss / max(len(loader.dataset) - len(loader), 1)
    return avg_loss, all_preds, all_labels, all_probs, resource, lat_ms


# ─────────────────────────────────────────────────────────────────────────────
# STAGE TRAINER
# ─────────────────────────────────────────────────────────────────────────────
def train_stage(bundle, unfreezer, train_loader, val_loader,
                stage, epochs, cfg, tracker, early_stopper, stage_log, device):
    model     = bundle.model
    optimizer = unfreezer.build_optimizer(stage)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    scaler    = GradScaler(device)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=max(epochs // 2, 5),
                                            T_mult=1, eta_min=1e-7)
    early_stopper.reset_for_new_stage(model)

    print(f"\n{'='*65}")
    print(f"  STAGE {stage} | Epochs: {epochs} | Trainable: {unfreezer.trainable_count():,}")
    print(f"{'='*65}")

    for epoch in range(1, epochs + 1):
        tr_loss,tr_p,tr_l,tr_pr,tr_res,tr_lat = run_epoch(
            model, train_loader, optimizer, criterion, scaler, device, "train")
        vl_loss,vl_p,vl_l,vl_pr,vl_res,vl_lat = run_epoch(
            model, val_loader,  optimizer, criterion, scaler, device, "val")
        scheduler.step()

        tr = tracker.compute(tr_loss, tr_p, tr_l, tr_pr)
        vl = tracker.compute(vl_loss, vl_p, vl_l, vl_pr)
        tracker.log("train", stage, epoch, tr, tr_res, tr_lat)
        tracker.log("val",   stage, epoch, vl, vl_res, vl_lat)

        row = {"stage": stage, "epoch": epoch,
               "train_acc": tr["acc"], "train_loss": tr["loss"],
               "val_acc": vl["acc"],   "val_loss":   vl["loss"],
               "val_f1":  vl["f1"],    "val_auc":    vl["auc"],
               **{f"val_{k}": v for k, v in vl_res.items()},
               "val_lat_ms": vl_lat}
        stage_log.append(row)

        print(f"  S{stage} Ep{epoch:03d} | "
              f"Tr Loss={tr['loss']:.4f} Acc={tr['acc']:.4f} | "
              f"Val Loss={vl['loss']:.4f} Acc={vl['acc']:.4f} "
              f"F1={vl['f1']:.4f} AUC={vl['auc']:.4f} | "
              f"GPU={vl_res['gpu_util_pct']:.0f}% "
              f"VRAM={vl_res['vram_mb']:.0f}MB "
              f"RAM={vl_res['ram_mb']:.0f}MB "
              f"Lat={vl_lat:.1f}ms")

        if early_stopper.step(vl["f1"], model):
            print(f"  ↳ Early stop at epoch {epoch} (best F1={early_stopper.best_score:.4f})")
            break

    early_stopper.restore(model)
    print(f"  ↳ Stage {stage} best Val F1 = {early_stopper.best_score:.4f}")
    return stage_log


# ─────────────────────────────────────────────────────────────────────────────
# HARDWARE PROFILER  ⏱ 💾
# ─────────────────────────────────────────────────────────────────────────────
class HardwareProfiler:

    @staticmethod
    def measure_model_load_time(model_name: str, num_classes: int,
                                dropout: float, device: str) -> float:
        """⏱ Time to build + load pretrained weights for this model."""
        t0 = time.perf_counter()
        ModelFactory.build(model_name, num_classes, dropout, device)
        return (time.perf_counter() - t0) * 1000  # ms

    @staticmethod
    def measure_inference_time(model, device, img_size=224, batch_size=32,
                               warmup=10, runs=50):
        """⏱ GPU/CPU inference time per image (ms) and throughput (img/s)."""
        model.eval()
        dummy = torch.randn(batch_size, 3, img_size, img_size).to(device)
        with torch.no_grad():
            for _ in range(warmup): model(dummy)
        if device == "cuda": torch.cuda.synchronize()
        times = []
        with torch.no_grad():
            for _ in range(runs):
                if device == "cuda": torch.cuda.synchronize()
                t0 = time.perf_counter()
                model(dummy)
                if device == "cuda": torch.cuda.synchronize()
                times.append((time.perf_counter() - t0) * 1000)
        batch_lat_ms = float(np.mean(times))
        per_img_ms   = batch_lat_ms / batch_size
        throughput   = batch_size * 1000 / batch_lat_ms
        return batch_lat_ms, per_img_ms, throughput

    @staticmethod
    def measure_feature_extraction_time(model, device, img_size=224,
                                        batch_size=32, runs=30) -> float:
        """
        ⏱ Time for backbone feature extraction only (excludes classifier head).
        Hooks into the classifier's first layer to stop early.
        Returns ms per image.
        """
        model.eval()
        dummy    = torch.randn(batch_size, 3, img_size, img_size).to(device)
        stop_flag = {"stop": False}
        times     = []

        class StopForwardException(Exception): pass
        def hook_fn(module, inp, out): raise StopForwardException()

        first_linear = None
        for m in model.modules():
            if isinstance(m, nn.Linear):
                first_linear = m; break
        if first_linear is None:
            return 0.0
        handle = first_linear.register_forward_hook(hook_fn)

        with torch.no_grad():
            for _ in range(runs):
                if device == "cuda": torch.cuda.synchronize()
                t0 = time.perf_counter()
                try: model(dummy)
                except StopForwardException: pass
                if device == "cuda": torch.cuda.synchronize()
                times.append((time.perf_counter() - t0) * 1000)
        handle.remove()
        return float(np.mean(times)) / batch_size  # ms per image

    @staticmethod
    def count_params_size(model):
        total     = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        size_mb   = sum(p.numel() * p.element_size()
                        for p in model.parameters()) / 1024**2
        flops = 0
        if THOP_AVAILABLE:
            try:
                dummy = torch.randn(1, 3, 224, 224).to(next(model.parameters()).device)
                macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
                flops = macs * 2
            except Exception:
                pass
        return total, trainable, size_mb, flops


# ─────────────────────────────────────────────────────────────────────────────
# FULL EXPERIMENT
# ─────────────────────────────────────────────────────────────────────────────
def run_experiment(model_name, train_loader, val_loader, test_loader,
                   class_names, cfg, ckpt_dir, save_dir):
    device = cfg["device"]
    print(f"\n{'#'*68}")
    print(f"  MODEL: {model_name.upper()}")
    print(f"{'#'*68}")

    # ── ⏱ Model load time ────────────────────────────────────────────────
    load_ms = HardwareProfiler.measure_model_load_time(
        model_name, cfg["num_classes"], cfg["dropout"], device)
    print(f"  ⏱  Model load time: {load_ms:.1f} ms")

    bundle     = ModelFactory.build(model_name, cfg["num_classes"], cfg["dropout"], device)
    unfreezer  = ProgressiveUnfreezer(bundle, cfg)
    tracker    = MetricsTracker(cfg["num_classes"])
    early_stop = EarlyStopping(cfg["early_stop_patience"], cfg["early_stop_delta"])
    stage_log  = []
    t_start    = time.time()

    for stage, epochs in [(1, cfg["stage1_epochs"]),
                          (2, cfg["stage2_epochs"]),
                          (3, cfg["stage3_epochs"])]:
        train_stage(bundle, unfreezer, train_loader, val_loader,
                    stage, epochs, cfg, tracker, early_stop, stage_log, device)

    train_time = time.time() - t_start

    # ── Test evaluation ────────────────────────────────────────────────────
    model = bundle.model; model.eval()
    criterion = nn.CrossEntropyLoss()
    scaler    = GradScaler(device)
    te_loss, te_p, te_l, te_pr, te_res, _ = run_epoch(
        model, test_loader, None, criterion, scaler, device, "val")
    test_metrics = tracker.compute(te_loss, te_p, te_l, te_pr)

    # ── ⏱ Inference timing ────────────────────────────────────────────────
    # CPU latency
    model_cpu = bundle.model.cpu()
    cpu_batch_lat, cpu_per_img_ms, _ = HardwareProfiler.measure_inference_time(
        model_cpu, "cpu", cfg["img_size"], min(8, cfg["batch_size"]), warmup=3, runs=10)
    bundle.model.to(device)

    gpu_batch_lat, gpu_per_img_ms, throughput = HardwareProfiler.measure_inference_time(
        model, device, cfg["img_size"], cfg["batch_size"])

    # ⏱ Feature extraction time
    feat_ms = HardwareProfiler.measure_feature_extraction_time(
        model, device, cfg["img_size"], cfg["batch_size"])

    total_p, trainable_p, size_mb, flops = HardwareProfiler.count_params_size(model)
    flops_per_image = flops / cfg["batch_size"] if flops > 0 else 0

    # 💾 Peak resource snapshot after test
    peak_res = ResourceMonitor.snapshot(device)

    results = {
        "model"              : model_name,
        # 📊 Performance
        "test_accuracy"      : test_metrics["acc"],
        "test_f1"            : test_metrics["f1"],
        "test_precision"     : test_metrics["prec"],
        "test_recall"        : test_metrics["rec"],
        "test_auc"           : test_metrics["auc"],
        "test_mcc"           : test_metrics["mcc"],
        "test_kappa"         : test_metrics["kappa"],
        "test_specificity"   : test_metrics["specificity"],
        "test_bal_acc"       : test_metrics["bal_acc"],
        "test_top5_acc"      : test_metrics["top5_acc"],
        "test_brier"         : test_metrics["brier"],
        # ⏱ Time
        "model_load_ms"      : load_ms,
        "gpu_batch_lat_ms"   : gpu_batch_lat,
        "gpu_per_img_ms"     : gpu_per_img_ms,
        "cpu_batch_lat_ms"   : cpu_batch_lat,
        "cpu_per_img_ms"     : cpu_per_img_ms,
        "feat_extract_ms"    : feat_ms,
        "throughput_img_s"   : throughput,
        "training_time_s"    : train_time,
        # 💾 Resource
        "gpu_util_pct"       : peak_res["gpu_util_pct"],
        "vram_mb"            : peak_res["vram_mb"],
        "ram_mb"             : peak_res["ram_mb"],
        # Model size
        "model_size_mb"      : size_mb,
        "flops"              : flops,
        "flops_per_image"    : flops_per_image,
        "total_params"       : total_p,
        "trainable_params"   : trainable_p,
        # Raw outputs for ensemble
        "test_preds"         : te_p,
        "test_labels"        : te_l,
        "test_probs"         : te_pr,
        "confusion_matrix"   : confusion_matrix(te_l, te_p).tolist(),
        "stage_log"          : stage_log,
        "cls_report"         : classification_report(
            te_l, te_p, target_names=class_names, output_dict=True),
    }

    # ── Save checkpoint ────────────────────────────────────────────────────
    torch.save(
        {"state_dict": model.state_dict(), "config": cfg,
         "metrics": {k: v for k, v in results.items()
                     if k not in ("confusion_matrix","stage_log","cls_report",
                                  "test_preds","test_labels","test_probs")}},
        ckpt_dir / f"{model_name}_best.pth")

    # ── Save soft probs + labels for ensemble reuse ────────────────────────
    np.save(save_dir / f"{model_name}_test_probs.npy",  np.array(te_pr))
    np.save(save_dir / f"{model_name}_test_labels.npy", np.array(te_l))
    print(f"  ✓ Ensemble arrays saved: "
          f"{model_name}_test_probs.npy  {model_name}_test_labels.npy")

    print(f"\n  ✓ {model_name} | "
          f"Acc={test_metrics['acc']:.4f} F1={test_metrics['f1']:.4f} "
          f"AUC={test_metrics['auc']:.4f} | "
          f"Infer={gpu_per_img_ms:.3f}ms/img | "
          f"FeatExt={feat_ms:.3f}ms/img | "
          f"LoadTime={load_ms:.1f}ms | "
          f"GPU={peak_res['gpu_util_pct']:.0f}% "
          f"VRAM={peak_res['vram_mb']:.0f}MB "
          f"RAM={peak_res['ram_mb']:.0f}MB")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
PALETTE = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b"]

def set_plot_style():
    plt.rcParams.update({
        "figure.dpi": 150, "savefig.dpi": 300,
        "font.family": "serif", "font.size": 10,
        "axes.labelsize": 11, "axes.titlesize": 12,
        "legend.fontsize": 9,
        "axes.spines.top": False, "axes.spines.right": False,
    })
set_plot_style()


def _stage_boundaries(ax, cfg):
    s1 = cfg["stage1_epochs"]; s2 = s1 + cfg["stage2_epochs"]
    for xv, lbl in [(s1,"Stage 2\n(+UPPER+TORSO)"), (s2,"Stage 3\n(Full Unfreeze)")]:
        ax.axvline(xv, color="gray", ls="--", lw=0.9, alpha=0.7)
        ax.text(xv+0.3, ax.get_ylim()[0]+0.02, lbl, fontsize=6.5,
                color="gray", va="bottom")


def plot_training_curves(all_results, cfg, save_dir):
    metrics = [("val_acc","Val Accuracy"),("val_loss","Val Loss"),
               ("val_f1","Val Macro-F1"),("val_auc","Val ROC-AUC")]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (m, lbl) in zip(axes.flatten(), metrics):
        for i,(mn,res) in enumerate(all_results.items()):
            df = pd.DataFrame(res["stage_log"])
            ax.plot(range(1,len(df)+1), df[m],
                    color=PALETTE[i%len(PALETTE)], label=mn, lw=1.8)
        ax.set_xlabel("Cumulative Epoch"); ax.set_ylabel(lbl)
        ax.set_title(lbl, fontweight="bold"); ax.legend(); ax.grid(True, alpha=0.25)
        _stage_boundaries(ax, cfg)
    plt.suptitle("Progressive Fine-Tuning: Learning Dynamics", fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = save_dir / "01_training_curves.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_comparative_bars(all_results, save_dir):
    models = list(all_results.keys())
    mets   = ["test_accuracy","test_f1","test_precision","test_recall","test_auc"]
    titles = ["Accuracy","Macro F1","Precision","Recall","AUC"]
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    x = np.arange(len(models))
    for ax, met, ttl in zip(axes, mets, titles):
        vals = [all_results[m][met] for m in models]
        bars = ax.bar(x, vals, color=PALETTE[:len(models)], edgecolor="white", width=0.6)
        ax.set_xticks(x); ax.set_xticklabels(models, rotation=35, ha="right")
        ax.set_ylim(0, 1.08); ax.set_title(ttl, fontweight="bold")
        ax.grid(axis="y", alpha=0.25)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=8)
    plt.suptitle("Classification Performance — All Models", fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = save_dir / "02_comparative_bars.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_confusion_matrices(all_results, class_names, save_dir):
    n = len(all_results); cols = min(3,n); rows = (n+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(7*cols, 6*rows))
    axes = np.array(axes).reshape(rows, cols)
    for idx,(mn,res) in enumerate(all_results.items()):
        ax = axes[idx//cols][idx%cols]
        cm = np.array(res["confusion_matrix"]).astype(float)
        cm /= cm.sum(axis=1, keepdims=True) + 1e-9
        tl = class_names if len(class_names) <= 15 else []
        sns.heatmap(cm, annot=len(class_names)<=15, fmt=".2f", cmap="Blues", ax=ax,
                    xticklabels=tl, yticklabels=tl, linewidths=0.4, vmin=0, vmax=1)
        ax.set_title(f"{mn}  (Acc={res['test_accuracy']:.4f})", fontweight="bold")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for idx in range(n, rows*cols):
        axes[idx//cols][idx%cols].axis("off")
    plt.suptitle("Normalized Confusion Matrices", fontsize=14, fontweight="bold")
    plt.tight_layout()
    out = save_dir / "03_confusion_matrices.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_time_metrics(all_results, save_dir):
    """⏱ Dedicated time-metrics chart."""
    models = list(all_results.keys())
    load   = [all_results[m]["model_load_ms"]    for m in models]
    infer  = [all_results[m]["gpu_per_img_ms"]   for m in models]
    feat   = [all_results[m]["feat_extract_ms"]  for m in models]
    cpu_i  = [all_results[m]["cpu_per_img_ms"]   for m in models]
    train  = [all_results[m]["training_time_s"]  for m in models]
    thru   = [all_results[m]["throughput_img_s"] for m in models]

    x = np.arange(len(models)); w = 0.22
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))

    def _bar(ax, vals, title, ylabel, color):
        bars = ax.bar(x, vals, color=color, edgecolor="white", width=0.55)
        ax.set_xticks(x); ax.set_xticklabels(models, rotation=30, ha="right")
        ax.set_title(title, fontweight="bold"); ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.25)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v*1.01,
                    f"{v:.2f}", ha="center", fontsize=8)

    _bar(axes[0,0], load,  "⏱ Model Load Time",          "ms",        "#3498db")
    _bar(axes[0,1], infer, "⏱ GPU Inference / Image",    "ms/img",    "#e74c3c")
    _bar(axes[0,2], feat,  "⏱ Feature Extraction / Image","ms/img",   "#9b59b6")
    _bar(axes[1,0], cpu_i, "⏱ CPU Inference / Image",    "ms/img",    "#f39c12")
    _bar(axes[1,1], train, "⏱ Total Training Time",      "seconds",   "#2ecc71")
    _bar(axes[1,2], thru,  "⏱ Inference Throughput",     "img/s",     "#1abc9c")

    plt.suptitle("Time Metrics Summary  ⏱", fontsize=14, fontweight="bold")
    plt.tight_layout()
    out = save_dir / "04a_time_metrics.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_resource_metrics(all_results, save_dir):
    """💾 GPU %, VRAM, RAM."""
    models  = list(all_results.keys())
    gpu_pct = [all_results[m]["gpu_util_pct"] for m in models]
    vram    = [all_results[m]["vram_mb"]       for m in models]
    ram     = [all_results[m]["ram_mb"]        for m in models]
    sz      = [all_results[m]["model_size_mb"] for m in models]

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    def _bar(ax, vals, title, ylabel, color):
        bars = ax.bar(np.arange(len(models)), vals, color=color,
                      edgecolor="white", width=0.6)
        ax.set_xticks(np.arange(len(models)))
        ax.set_xticklabels(models, rotation=30, ha="right")
        ax.set_title(title, fontweight="bold"); ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.25)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v*1.01,
                    f"{v:.1f}", ha="center", fontsize=8)

    _bar(axes[0], gpu_pct, "💾 GPU Utilisation",  "%",  "#e74c3c")
    _bar(axes[1], vram,    "💾 VRAM Usage",        "MB", "#3498db")
    _bar(axes[2], ram,     "💾 RAM Footprint",     "MB", "#2ecc71")
    _bar(axes[3], sz,      "💾 Model Size on Disk","MB", "#9b59b6")

    plt.suptitle("Resource Metrics Summary  💾", fontsize=14, fontweight="bold")
    plt.tight_layout()
    out = save_dir / "04b_resource_metrics.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_efficiency(all_results, save_dir):
    models = list(all_results.keys())
    acc    = [all_results[m]["test_accuracy"]    for m in models]
    lat    = [all_results[m]["gpu_per_img_ms"]   for m in models]
    mem    = [all_results[m]["vram_mb"]          for m in models]
    sz     = [all_results[m]["model_size_mb"]    for m in models]
    thru   = [all_results[m]["throughput_img_s"] for m in models]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, m in enumerate(models):
        axes[0].scatter(lat[i], acc[i], s=sz[i]*0.6,
                        color=PALETTE[i%len(PALETTE)], alpha=0.85, edgecolors="k", lw=0.5)
        axes[0].annotate(m, (lat[i],acc[i]), textcoords="offset points", xytext=(6,3), fontsize=8)
        axes[1].scatter(mem[i], acc[i], s=120,
                        color=PALETTE[i%len(PALETTE)], alpha=0.85, edgecolors="k", lw=0.5, label=m)
        axes[1].annotate(m, (mem[i],acc[i]), textcoords="offset points", xytext=(6,3), fontsize=8)
    axes[0].set_xlabel("GPU Latency (ms/img)"); axes[0].set_ylabel("Test Accuracy")
    axes[0].set_title("Accuracy vs. GPU Latency\n(bubble ∝ model size)"); axes[0].grid(True, alpha=0.25)
    axes[1].set_xlabel("VRAM Usage (MB)"); axes[1].set_ylabel("Test Accuracy")
    axes[1].set_title("Accuracy vs. VRAM"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.25)
    x = np.arange(len(models))
    axes[2].bar(x, thru, color=PALETTE[:len(models)], edgecolor="white")
    axes[2].set_xticks(x); axes[2].set_xticklabels(models, rotation=30, ha="right")
    axes[2].set_ylabel("Throughput (img/s)"); axes[2].set_title("Inference Throughput")
    axes[2].grid(axis="y", alpha=0.25)
    plt.suptitle("Efficiency Analysis", fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = save_dir / "04c_efficiency.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_stage_progression(all_results, save_dir):
    fig, ax = plt.subplots(figsize=(12, 6))
    for i,(mn,res) in enumerate(all_results.items()):
        df = pd.DataFrame(res["stage_log"]); pts = []
        for s in [1,2,3]:
            sub = df[df["stage"]==s]
            if not sub.empty: pts.append((s, sub["val_acc"].max()))
        pts.append((3.6, res["test_accuracy"]))
        xs, ys = zip(*pts)
        ax.plot(xs, ys, "o-", color=PALETTE[i%len(PALETTE)], label=mn, lw=2, ms=8)
    ax.set_xticks([1,2,3,3.6])
    ax.set_xticklabels(["Stage 1\n(HEAD)","Stage 2\n(+UPPER+TORSO)","Stage 3\n(Full)","Test"])
    ax.set_ylabel("Accuracy"); ax.set_title("Accuracy Lift per Fine-Tuning Stage", fontweight="bold")
    ax.legend(loc="lower right"); ax.grid(True, alpha=0.25); ax.axvline(3.6, color="red", ls=":", alpha=0.5)
    plt.tight_layout()
    out = save_dir / "05_stage_progression.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def save_summary_table(all_results, save_dir):
    rows = []
    for mn, res in all_results.items():
        rows.append({
            "Model"              : mn,
            # 📊 Performance
            "Accuracy"           : f"{res['test_accuracy']:.4f}",
            "Balanced Acc"       : f"{res['test_bal_acc']:.4f}",
            "Macro F1"           : f"{res['test_f1']:.4f}",
            "Precision"          : f"{res['test_precision']:.4f}",
            "Recall"             : f"{res['test_recall']:.4f}",
            "AUC"                : f"{res['test_auc']:.4f}",
            "MCC"                : f"{res['test_mcc']:.4f}",
            "Kappa"              : f"{res['test_kappa']:.4f}",
            "Specificity"        : f"{res['test_specificity']:.4f}",
            "Top-5 Acc"          : f"{res['test_top5_acc']:.4f}",
            "Brier Score"        : f"{res['test_brier']:.4f}",
            # ⏱ Time
            "Load (ms)"          : f"{res['model_load_ms']:.1f}",
            "GPU Infer/img (ms)" : f"{res['gpu_per_img_ms']:.3f}",
            "CPU Infer/img (ms)" : f"{res['cpu_per_img_ms']:.3f}",
            "FeatExt/img (ms)"   : f"{res['feat_extract_ms']:.3f}",
            "Throughput (img/s)" : f"{res['throughput_img_s']:.1f}",
            "Train Time (s)"     : f"{res['training_time_s']:.1f}",
            # 💾 Resource
            "GPU Util (%)"       : f"{res['gpu_util_pct']:.1f}",
            "VRAM (MB)"          : f"{res['vram_mb']:.1f}",
            "RAM (MB)"           : f"{res['ram_mb']:.1f}",
            "Model MB"           : f"{res['model_size_mb']:.1f}",
            "GFLOPs/img"         : f"{res['flops_per_image']/1e9:.3f}",
            "Params (M)"         : f"{res['total_params']/1e6:.2f}",
        })
    df = pd.DataFrame(rows)

    acc   = np.array([res["test_accuracy"] for res in all_results.values()])
    f1s   = np.array([res["test_f1"]       for res in all_results.values()])
    mccs  = np.array([res["test_mcc"]      for res in all_results.values()])
    lats  = np.array([res["gpu_per_img_ms"]for res in all_results.values()])
    lat_n = 1 - (lats - lats.min()) / (lats.max() - lats.min() + 1e-9)
    composite = 0.35*acc + 0.25*f1s + 0.20*((mccs+1)/2) + 0.20*lat_n
    best_idx  = int(np.argmax(composite))

    print("\n" + "="*120)
    print("SUMMARY TABLE"); print("="*120)
    print(df.to_string(index=False)); print("="*120)
    df.to_csv(save_dir / "summary_table.csv", index=False)

    fig, ax = plt.subplots(figsize=(38, max(3, len(rows)*0.7 + 2)))
    ax.axis("off")
    tbl = ax.table(cellText=df.values, colLabels=df.columns, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(6.5); tbl.scale(1.1, 1.6)
    for j in range(len(df.columns)):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
        tbl[best_idx+1, j].set_facecolor("#d5f5e3")
    ax.set_title(
        f"IEEE Comparative Results — Best: {list(all_results.keys())[best_idx].upper()} "
        f"(composite={composite[best_idx]:.4f})",
        fontsize=11, fontweight="bold", pad=12)
    plt.savefig(save_dir / "06_summary_table.png", bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Table saved → {save_dir / 'summary_table.csv'}")
    return df, list(all_results.keys())[best_idx], composite[best_idx]


def plot_per_class_f1(all_results, class_names, save_dir):
    data = {}
    for mn,res in all_results.items():
        report = res["cls_report"]
        data[mn] = [report.get(c,{}).get("f1-score",0.0) for c in class_names]
    df = pd.DataFrame(data, index=class_names)
    fig, ax = plt.subplots(figsize=(max(10,len(all_results)*2+2), max(8,len(class_names)*0.4+2)))
    sns.heatmap(df, annot=True, fmt=".2f", cmap="RdYlGn", ax=ax, linewidths=0.5, vmin=0, vmax=1)
    ax.set_title("Per-Class F1-Score Heatmap", fontweight="bold")
    ax.set_xlabel("Model"); ax.set_ylabel("Malware Family")
    plt.tight_layout()
    out = save_dir / "07_per_class_f1.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_roc_curves(all_results, class_names, save_dir):
    fig, ax = plt.subplots(figsize=(8, 7))
    for i,(mn,res) in enumerate(all_results.items()):
        labels = np.array(res["test_labels"]); probs = np.array(res["test_probs"])
        n_cls  = probs.shape[1]
        lb     = label_binarize(labels, classes=list(range(n_cls)))
        all_fpr = np.linspace(0,1,200); tprs = []
        for c in range(n_cls):
            try:
                fpr, tpr, _ = roc_curve(lb[:,c], probs[:,c])
                tprs.append(np.interp(all_fpr, fpr, tpr))
            except Exception: pass
        if tprs:
            mean_tpr = np.mean(tprs, axis=0)
            ax.plot(all_fpr, mean_tpr, color=PALETTE[i%len(PALETTE)], lw=1.8,
                    label=f"{mn} (AUC={res['test_auc']:.3f})")
    ax.plot([0,1],[0,1],"k--",lw=0.8,alpha=0.5,label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("Macro-Average ROC Curves — All Models", fontweight="bold")
    ax.legend(loc="lower right", fontsize=8, frameon=True); ax.grid(True, alpha=0.25)
    plt.tight_layout()
    out = save_dir / "08_roc_curves.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_calibration(all_results, save_dir):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0,1],[0,1],"k--",lw=0.9,alpha=0.6,label="Perfect calibration")
    for i,(mn,res) in enumerate(all_results.items()):
        labels = np.array(res["test_labels"]); probs = np.array(res["test_probs"])
        confidence  = probs.max(axis=1)
        correctness = (probs.argmax(axis=1) == labels).astype(float)
        try:
            frac_pos, mean_pred = calibration_curve(correctness, confidence, n_bins=10, strategy="uniform")
            ax.plot(mean_pred, frac_pos, "o-", color=PALETTE[i%len(PALETTE)], lw=1.6, ms=5,
                    label=f"{mn} (Brier={res['test_brier']:.3f})")
        except Exception: pass
    ax.set_xlabel("Mean Predicted Confidence"); ax.set_ylabel("Fraction Correct")
    ax.set_title("Reliability Diagram (Calibration)", fontweight="bold")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
    plt.tight_layout()
    out = save_dir / "09_calibration.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_tsne(model, test_loader, class_names, device, save_dir, model_name, max_samples=2000):
    model.eval(); features_list, labels_list = [], []
    hook_out = {}
    def hook_fn(module, inp, out): hook_out["feat"] = inp[0].detach().cpu()
    last_linear = None
    for m in model.modules():
        if isinstance(m, nn.Linear): last_linear = m
    if last_linear is None: print("  t-SNE: no Linear found, skip."); return
    handle = last_linear.register_forward_hook(hook_fn)
    with torch.no_grad():
        for X, y in test_loader:
            model(X.to(device)); features_list.append(hook_out["feat"].numpy())
            labels_list.extend(y.tolist())
            if sum(len(f) for f in features_list) >= max_samples: break
    handle.remove()
    feats = np.concatenate(features_list)[:max_samples]
    labels= np.array(labels_list)[:max_samples]
    print(f"  t-SNE: {len(feats)} samples …")
    emb   = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42, n_jobs=-1).fit_transform(feats)
    fig, ax = plt.subplots(figsize=(10, 8))
    n_cls = len(class_names); cmap = plt.cm.get_cmap("tab20", n_cls)
    for c in range(n_cls):
        mask = labels == c
        if mask.sum() == 0: continue
        ax.scatter(emb[mask,0], emb[mask,1], s=12, alpha=0.7, color=cmap(c),
                   label=class_names[c] if n_cls<=15 else None)
    ax.set_title(f"t-SNE Feature Space — {model_name}", fontweight="bold")
    ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
    if n_cls <= 15: ax.legend(fontsize=7, ncol=2, frameon=True)
    ax.grid(True, alpha=0.2); plt.tight_layout()
    out = save_dir / f"10_tsne_{model_name}.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_radar(all_results, save_dir):
    metrics = ["test_accuracy","test_f1","test_auc","test_mcc","test_specificity","test_bal_acc"]
    labels  = ["Accuracy","Macro F1","ROC-AUC","MCC","Specificity","Bal. Acc"]
    N = len(metrics); angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist(); angles += angles[:1]
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for i,(mn,res) in enumerate(all_results.items()):
        vals = [float(np.clip((res[m]+1)/2 if m=="test_mcc" else res[m], 0, 1)) for m in metrics]
        vals += vals[:1]
        ax.plot(angles, vals, color=PALETTE[i%len(PALETTE)], lw=2, label=mn)
        ax.fill(angles, vals, color=PALETTE[i%len(PALETTE)], alpha=0.07)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
    ax.set_ylim(0,1); ax.set_title("Radar Chart\n(MCC normalized to [0,1])", fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35,1.1), fontsize=8)
    plt.tight_layout()
    out = save_dir / "11_radar_chart.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_pareto(all_results, save_dir):
    models = list(all_results.keys())
    acc    = np.array([all_results[m]["test_accuracy"]   for m in models])
    flops  = np.array([all_results[m]["flops_per_image"] for m in models])
    sz     = np.array([all_results[m]["model_size_mb"]   for m in models])
    pareto = [i for i in range(len(models))
              if not any(acc[j]>=acc[i] and flops[j]<=flops[i] and
                         (acc[j]>acc[i] or flops[j]<flops[i])
                         for j in range(len(models)) if j!=i)]
    fig, ax = plt.subplots(figsize=(10, 7))
    for i,mn in enumerate(models):
        mk = "*" if i in pareto else "o"; ms = 200 if i in pareto else 80
        ax.scatter(flops[i]/1e9, acc[i], s=ms, color=PALETTE[i%len(PALETTE)],
                   marker=mk, edgecolors="k", lw=0.7, zorder=3)
        ax.annotate(mn, (flops[i]/1e9, acc[i]), textcoords="offset points", xytext=(7,4), fontsize=8)
    pf = sorted(pareto, key=lambda i: flops[i])
    if len(pf) > 1:
        ax.plot([flops[i]/1e9 for i in pf],[acc[i] for i in pf],
                "r--",lw=1.4,alpha=0.8,label="Pareto frontier")
    ax.set_xlabel("GFLOPs/img"); ax.set_ylabel("Test Accuracy")
    ax.set_title("Pareto Frontier: Accuracy vs. Compute\n(★ = Pareto-optimal)", fontweight="bold")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
    plt.tight_layout()
    out = save_dir / "12_pareto_frontier.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


def plot_statistical_test(all_results, save_dir):
    metric_map = {"Accuracy":"test_accuracy","Macro F1":"test_f1","AUC":"test_auc",
                  "MCC":"test_mcc","Specificity":"test_specificity","Bal. Acc":"test_bal_acc"}
    models = list(all_results.keys())
    score_mat = np.array([[all_results[m][v] for m in models] for v in metric_map.values()])
    rank_mat  = np.zeros_like(score_mat)
    for i in range(len(score_mat)):
        for rank, idx in enumerate(np.argsort(-score_mat[i])): rank_mat[i,idx] = rank+1
    mean_ranks = rank_mat.mean(axis=0)
    try: stat, p_val = friedmanchisquare(*score_mat.T.tolist())
    except Exception: stat, p_val = 0.0, 1.0
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    bars = axes[0].bar(models, mean_ranks, color=[PALETTE[i%len(PALETTE)] for i in range(len(models))], edgecolor="white")
    axes[0].set_xticklabels(models, rotation=35, ha="right")
    axes[0].set_ylabel("Mean Rank (lower = better)")
    axes[0].set_title(f"Friedman Ranks\nχ²={stat:.2f}, p={p_val:.4f} "
                      f"({'significant' if p_val<0.05 else 'not significant'})", fontweight="bold")
    axes[0].grid(axis="y", alpha=0.25)
    for bar, v in zip(bars, mean_ranks):
        axes[0].text(bar.get_x()+bar.get_width()/2, v+0.05, f"{v:.2f}", ha="center", fontsize=8)
    if POSTHOCS_AVAILABLE and p_val < 0.05:
        try:
            ph_df = posthoc_nemenyi_friedman(score_mat.T)
            ph_df.index = models; ph_df.columns = models
            sns.heatmap(ph_df, annot=True, fmt=".3f", cmap="RdYlGn_r",
                        ax=axes[1], vmin=0, vmax=0.1, linewidths=0.5)
            axes[1].set_title("Nemenyi Post-Hoc p-values", fontweight="bold")
        except Exception as e:
            axes[1].text(0.5,0.5,f"Nemenyi failed:\n{e}",ha="center",va="center",transform=axes[1].transAxes)
    else:
        msg = ("pip install scikit-posthocs" if not POSTHOCS_AVAILABLE
               else "Friedman p ≥ 0.05:\nNo significant differences")
        axes[1].text(0.5,0.5,msg,ha="center",va="center",transform=axes[1].transAxes,
                     fontsize=11,bbox=dict(boxstyle="round",facecolor="#f0f0f0"))
        axes[1].axis("off")
    plt.suptitle("Statistical Significance (Friedman + Nemenyi)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = save_dir / "13_statistical_test.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")
    pd.DataFrame(rank_mat.T, index=models, columns=list(metric_map.keys())
                 ).assign(**{"Mean Rank": mean_ranks}).to_csv(save_dir / "friedman_ranks.csv")


def plot_mcc_kappa(all_results, save_dir):
    models = list(all_results.keys())
    mccs   = [all_results[m]["test_mcc"]   for m in models]
    kappas = [all_results[m]["test_kappa"] for m in models]
    x = np.arange(len(models)); w = 0.35
    fig, ax = plt.subplots(figsize=(13, 5))
    b1 = ax.bar(x-w/2, mccs,   w, label="MCC",          color="#2ecc71", edgecolor="white")
    b2 = ax.bar(x+w/2, kappas, w, label="Cohen's Kappa", color="#3498db", edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=30, ha="right")
    ax.set_ylim(-0.1, 1.1); ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_ylabel("Score"); ax.set_title("MCC vs. Cohen's Kappa", fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.25)
    for bar, v in zip(list(b1)+list(b2), mccs+kappas):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=8)
    plt.tight_layout()
    out = save_dir / "14_mcc_kappa.png"
    plt.savefig(out, bbox_inches="tight"); plt.close(); print(f"  Saved → {out}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir  = Path(CONFIG["save_dir"]) / timestamp
    ckpt_dir  = Path(CONFIG["checkpoint_dir"])
    save_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    with open(save_dir / "config.json", "w") as f:
        json.dump({**CONFIG, "drive_config": DRIVE_CONFIG}, f, indent=2)

    print(f"\n{'='*65}")
    print(f"  Experiment → {save_dir}")
    print(f"  Device: {CONFIG['device'].upper()} | AMP: {CONFIG['use_amp']} | Seed: {CONFIG['seed']}")
    if DRIVE_CONFIG["SAVE_TO_DRIVE"]:
        subfolder = DRIVE_CONFIG["DRIVE_SUBFOLDER"] or timestamp
        print(f"  Drive path → MyDrive/{DRIVE_CONFIG['DRIVE_FOLDER']}/{subfolder}")
    print(f"{'='*65}")

    train_loader, val_loader, test_loader, class_names = build_dataloaders(CONFIG)
    CONFIG["num_classes"] = len(class_names)
    print(f"  Classes: {CONFIG['num_classes']} | "
          f"Train batches: {len(train_loader)} | Val: {len(val_loader)}")

    all_results = {}
    for model_name in MODELS_TO_RUN:
        try:
            res = run_experiment(model_name, train_loader, val_loader,
                                 test_loader, class_names, CONFIG, ckpt_dir, save_dir)
            all_results[model_name] = res

            if DRIVE_CONFIG["SAVE_TO_DRIVE"] and DRIVE_CONFIG["INCREMENTAL_SAVE"]:
                print(f"\n  [Drive] Incremental backup after {model_name} …")
                mount_and_save_to_drive(save_dir)

            # Save partial results.json (excluding large raw arrays)
            safe = {m: {k: v for k, v in r.items()
                        if k not in ("cls_report","test_preds","test_labels","test_probs")}
                    for m, r in all_results.items()}
            with open(save_dir / "results.json", "w") as f:
                json.dump(safe, f, indent=2, default=str)

        except Exception as e:
            import traceback
            print(f"  ERROR {model_name}: {e}"); traceback.print_exc()

    if not all_results:
        print("No models completed."); return

    print("\nGenerating plots …")
    plot_training_curves(all_results, CONFIG, save_dir)
    plot_comparative_bars(all_results, save_dir)
    plot_confusion_matrices(all_results, class_names, save_dir)
    plot_time_metrics(all_results, save_dir)          # ⏱ new
    plot_resource_metrics(all_results, save_dir)      # 💾 new
    plot_efficiency(all_results, save_dir)
    plot_stage_progression(all_results, save_dir)
    df, best_model, best_score = save_summary_table(all_results, save_dir)
    plot_per_class_f1(all_results, class_names, save_dir)
    plot_roc_curves(all_results, class_names, save_dir)
    plot_calibration(all_results, save_dir)
    plot_radar(all_results, save_dir)
    plot_pareto(all_results, save_dir)
    plot_statistical_test(all_results, save_dir)
    plot_mcc_kappa(all_results, save_dir)

    # t-SNE for best model only
    print(f"\n  Running t-SNE for best model ({best_model}) …")
    try:
        best_bundle = ModelFactory.build(
            best_model, CONFIG["num_classes"], CONFIG["dropout"], CONFIG["device"])
        best_bundle.model.load_state_dict(
            torch.load(ckpt_dir / f"{best_model}_best.pth",
                       map_location=CONFIG["device"])["state_dict"])
        plot_tsne(best_bundle.model, test_loader, class_names,
                  CONFIG["device"], save_dir, best_model)
    except Exception as e:
        print(f"  t-SNE skipped: {e}")

    # Final Drive save
    if DRIVE_CONFIG["SAVE_TO_DRIVE"]:
        print("\n  Final save to Google Drive …")
        mount_and_save_to_drive(save_dir)

    print(f"\n{'='*65}")
    print(f"  ★  BEST MODEL: {best_model.upper()}")
    r = all_results[best_model]
    print(f"  📊 Accuracy    : {r['test_accuracy']:.4f}")
    print(f"  📊 Macro F1    : {r['test_f1']:.4f}")
    print(f"  📊 ROC-AUC     : {r['test_auc']:.4f}")
    print(f"  📊 Precision   : {r['test_precision']:.4f}")
    print(f"  📊 Recall      : {r['test_recall']:.4f}")
    print(f"  ⏱  Load Time   : {r['model_load_ms']:.1f} ms")
    print(f"  ⏱  Infer/img   : {r['gpu_per_img_ms']:.3f} ms")
    print(f"  ⏱  FeatExt/img : {r['feat_extract_ms']:.3f} ms")
    print(f"  ⏱  Throughput  : {r['throughput_img_s']:.1f} img/s")
    print(f"  💾 GPU Util    : {r['gpu_util_pct']:.1f} %")
    print(f"  💾 VRAM        : {r['vram_mb']:.1f} MB")
    print(f"  💾 RAM         : {r['ram_mb']:.1f} MB")
    print(f"  💾 Model Size  : {r['model_size_mb']:.1f} MB")
    print(f"  Composite Score: {best_score:.4f}")
    print(f"\n  Local outputs → {save_dir}")
    if DRIVE_CONFIG["SAVE_TO_DRIVE"]:
        subfolder = DRIVE_CONFIG["DRIVE_SUBFOLDER"] or timestamp
        print(f"  Drive copy    → MyDrive/{DRIVE_CONFIG['DRIVE_FOLDER']}/{subfolder}")
    print(f"\n  Ensemble arrays saved per model:")
    for mn in all_results:
        print(f"    {save_dir}/{mn}_test_probs.npy   {save_dir}/{mn}_test_labels.npy")
    print(f"{'='*65}")


if __name__ == "__main__":
    main()


  Experiment → experiment_results/20260509_135427
  Device: CUDA | AMP: True | Seed: 42
  Drive path → MyDrive/MalwareExperiments/20260509_135427
  Classes: 25 | Train batches: 233 | Val: 28

####################################################################
  MODEL: RESNET50
####################################################################
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s]


  Module split → BASE: 3 | TORSO: 3 | UPPER: 4
    BASE: ['conv1', 'bn1', 'relu']
    TORSO: ['maxpool', 'layer1', 'layer2']
    UPPER: ['layer3', 'layer4', 'avgpool', 'fc']
  ⏱  Model load time: 1378.3 ms
  Module split → BASE: 3 | TORSO: 3 | UPPER: 4
    BASE: ['conv1', 'bn1', 'relu']
    TORSO: ['maxpool', 'layer1', 'layer2']
    UPPER: ['layer3', 'layer4', 'avgpool', 'fc']
  Stage 1 → Groups: ['classifier'] | Trainable: 1,067,033

  STAGE 1 | Epochs: 15 | Trainable: 1,067,033
  S1 Ep001 | Tr Loss=1.3143 Acc=0.7536 | Val Loss=1.5109 Acc=0.6752 F1=0.7374 AUC=0.9884 | GPU=36% VRAM=1066MB RAM=2025MB Lat=139.0ms
  S1 Ep002 | Tr Loss=1.1177 Acc=0.8246 | Val Loss=1.8878 Acc=0.6194 F1=0.7200 AUC=0.9855 | GPU=26% VRAM=1206MB RAM=2039MB Lat=202.2ms
  S1 Ep003 | Tr Loss=1.0758 Acc=0.8428 | Val Loss=1.6602 Acc=0.6585 F1=0.7597 AUC=0.9904 | GPU=59% VRAM=1206MB RAM=2040MB Lat=202.8ms
  S1 Ep004 | Tr Loss=1.0428 Acc=0.8564 | Val Loss=1.3233 Acc=0.7299 F1=0.7785 AUC=0.9920 | GPU=39% VRAM=1206MB RA

100%|██████████| 171M/171M [00:01<00:00, 118MB/s]


  Module split → BASE: 3 | TORSO: 3 | UPPER: 4
    BASE: ['conv1', 'bn1', 'relu']
    TORSO: ['maxpool', 'layer1', 'layer2']
    UPPER: ['layer3', 'layer4', 'avgpool', 'fc']
  ⏱  Model load time: 2781.0 ms
  Module split → BASE: 3 | TORSO: 3 | UPPER: 4
    BASE: ['conv1', 'bn1', 'relu']
    TORSO: ['maxpool', 'layer1', 'layer2']
    UPPER: ['layer3', 'layer4', 'avgpool', 'fc']
  Stage 1 → Groups: ['classifier'] | Trainable: 1,067,033

  STAGE 1 | Epochs: 15 | Trainable: 1,067,033
  S1 Ep001 | Tr Loss=1.3311 Acc=0.7507 | Val Loss=1.4552 Acc=0.7054 F1=0.7044 AUC=0.9879 | GPU=89% VRAM=4522MB RAM=2400MB Lat=188.5ms
  S1 Ep002 | Tr Loss=1.1379 Acc=0.8132 | Val Loss=1.5592 Acc=0.6417 F1=0.6679 AUC=0.9697 | GPU=67% VRAM=4568MB RAM=2401MB Lat=148.6ms
  S1 Ep003 | Tr Loss=1.0874 Acc=0.8369 | Val Loss=1.4504 Acc=0.6864 F1=0.7334 AUC=0.9889 | GPU=99% VRAM=4568MB RAM=2401MB Lat=175.6ms
  S1 Ep004 | Tr Loss=1.0516 Acc=0.8515 | Val Loss=1.3700 Acc=0.7020 F1=0.7699 AUC=0.9929 | GPU=88% VRAM=4568MB RA

...

  Module split → BASE: 4 | TORSO: 4 | UPPER: 4
    BASE: ['conv0', 'norm0', 'relu0', 'pool0']
    TORSO: ['denseblock1', 'transition1', 'denseblock2', 'transition2']
    UPPER: ['denseblock3', 'transition3', 'denseblock4', 'norm5']
  ⏱  Model load time: 547.1 ms
  Module split → BASE: 4 | TORSO: 4 | UPPER: 4
    BASE: ['conv0', 'norm0', 'relu0', 'pool0']
    TORSO: ['denseblock1', 'transition1', 'denseblock2', 'transition2']
    UPPER: ['denseblock3', 'transition3', 'denseblock4', 'norm5']
  Stage 1 → Groups: ['classifier'] | Trainable: 540,697

  STAGE 1 | Epochs: 15 | Trainable: 540,697
  S1 Ep001 | Tr Loss=1.1974 Acc=0.7945 | Val Loss=1.1238 Acc=0.7667 F1=0.8064 AUC=0.9971 | GPU=75% VRAM=5814MB RAM=2404MB Lat=218.1ms
  S1 Ep002 | Tr Loss=1.0116 Acc=0.8707 | Val Loss=1.1036 Acc=0.7712 F1=0.7578 AUC=0.9972 | GPU=72% VRAM=5814MB RAM=2405MB Lat=150.4ms
  S1 Ep003 | Tr Loss=0.9740 Acc=0.8737 | Val Loss=1.1608 Acc=0.7667 F1=0.7983 AUC=0.9975 | GPU=22% VRAM=5814MB RAM=2405MB Lat=147.8ms
  

100%|██████████| 13.6M/13.6M [00:00<00:00, 154MB/s]


  Module split → BASE: 6 | TORSO: 6 | UPPER: 7
    BASE: ['0', '1', '2', '3', '4', '5']
    TORSO: ['6', '7', '8', '9', '10', '11']
    UPPER: ['12', '13', '14', '15', '16', '17', '18']
  ⏱  Model load time: 240.6 ms
  Module split → BASE: 6 | TORSO: 6 | UPPER: 7
    BASE: ['0', '1', '2', '3', '4', '5']
    TORSO: ['6', '7', '8', '9', '10', '11']
    UPPER: ['12', '13', '14', '15', '16', '17', '18']
  Stage 1 → Groups: ['classifier'] | Trainable: 672,281

  STAGE 1 | Epochs: 15 | Trainable: 672,281
  S1 Ep001 | Tr Loss=1.3416 Acc=0.7454 | Val Loss=2.1101 Acc=0.5848 F1=0.6751 AUC=0.9774 | GPU=12% VRAM=5816MB RAM=2474MB Lat=201.5ms
  S1 Ep002 | Tr Loss=1.1481 Acc=0.8055 | Val Loss=1.6725 Acc=0.6016 F1=0.6917 AUC=0.9784 | GPU=50% VRAM=5816MB RAM=2474MB Lat=212.5ms
  S1 Ep003 | Tr Loss=1.1217 Acc=0.8207 | Val Loss=1.6882 Acc=0.6150 F1=0.7130 AUC=0.9810 | GPU=40% VRAM=5816MB RAM=2474MB Lat=185.2ms
  S1 Ep004 | Tr Loss=1.0869 Acc=0.8322 | Val Loss=1.6028 Acc=0.6429 F1=0.7387 AUC=0.9819 | GPU

model.safetensors:   0%|          | 0.00/224M [00:00<?, ?B/s]

  Module split → BASE: 6 | TORSO: 6 | UPPER: 6
    BASE: ['conv2d_1a', 'conv2d_2a', 'conv2d_2b', 'maxpool_3a', 'conv2d_3b', 'conv2d_4a']
    TORSO: ['maxpool_5a', 'mixed_5b', 'repeat', 'mixed_6a', 'repeat_1', 'mixed_7a']
    UPPER: ['repeat_2', 'block8', 'conv2d_7b', 'global_pool', 'head_drop', 'classif']
  ⏱  Model load time: 18407.4 ms
  Module split → BASE: 6 | TORSO: 6 | UPPER: 6
    BASE: ['conv2d_1a', 'conv2d_2a', 'conv2d_2b', 'maxpool_3a', 'conv2d_3b', 'conv2d_4a']
    TORSO: ['maxpool_5a', 'mixed_5b', 'repeat', 'mixed_6a', 'repeat_1', 'mixed_7a']
    UPPER: ['repeat_2', 'block8', 'conv2d_7b', 'global_pool', 'head_drop', 'classif']
  Stage 1 → Groups: ['classifier'] | Trainable: 803,865

  STAGE 1 | Epochs: 15 | Trainable: 803,865
  S1 Ep001 | Tr Loss=1.5580 Acc=0.6655 | Val Loss=1.5694 Acc=0.6641 F1=0.6811 AUC=0.9895 | GPU=90% VRAM=6112MB RAM=2783MB Lat=166.2ms
  S1 Ep002 | Tr Loss=1.3263 Acc=0.7528 | Val Loss=1.3984 Acc=0.7065 F1=0.6911 AUC=0.9892 | GPU=75% VRAM=6112MB RAM=279

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
